# CWRU Bearing Dataset — Notebook 1: Download

Downloads **all** Drive-End bearing fault data from CWRU servers:
- ✅ Normal baseline (4 loads)
- ✅ 12k Drive-End: IR, Ball, OR@6, OR@3, OR@12 — all severities + 0.028"
- ✅ 48k Drive-End: IR, Ball, OR@6, OR@3, OR@12 — all severities

**Run once.** Skips files already downloaded. Retries with backoff on network errors.

Output: `cwru_data/` folder with all `.mat` files + `cwru_manifest.csv` metadata table.

In [ ]:
import os, time, requests, csv
from tqdm.notebook import tqdm

DATA_DIR = 'cwru_data'
os.makedirs(DATA_DIR, exist_ok=True)

BASE = 'https://engineering.case.edu/sites/default/files/'

# ─────────────────────────────────────────────────────────────────────────────
# FILE REGISTRY
# Columns: (filename, file_id, sample_rate, fault_type, fault_loc, severity_mils, load_hp, rpm)
# fault_loc: IR | OR@6 | OR@3 | OR@12 | Ball | Normal
# ─────────────────────────────────────────────────────────────────────────────

FILES = [

    # ── Normal baseline (shared for both rates) ───────────────────────────
    ('Normal_0.mat',      '97',   12000, 'Normal', 'Normal',  0,  0, 1797),
    ('Normal_1.mat',      '98',   12000, 'Normal', 'Normal',  0,  1, 1772),
    ('Normal_2.mat',      '99',   12000, 'Normal', 'Normal',  0,  2, 1750),
    ('Normal_3.mat',      '100',  12000, 'Normal', 'Normal',  0,  3, 1730),

    # ── 12k Drive-End — Inner Race ────────────────────────────────────────
    ('12k_IR007_0.mat',   '105',  12000, 'IR',  'IR',   7,  0, 1797),
    ('12k_IR007_1.mat',   '106',  12000, 'IR',  'IR',   7,  1, 1772),
    ('12k_IR007_2.mat',   '107',  12000, 'IR',  'IR',   7,  2, 1750),
    ('12k_IR007_3.mat',   '108',  12000, 'IR',  'IR',   7,  3, 1730),
    ('12k_IR014_0.mat',   '169',  12000, 'IR',  'IR',  14,  0, 1797),
    ('12k_IR014_1.mat',   '170',  12000, 'IR',  'IR',  14,  1, 1772),
    ('12k_IR014_2.mat',   '171',  12000, 'IR',  'IR',  14,  2, 1750),
    ('12k_IR014_3.mat',   '172',  12000, 'IR',  'IR',  14,  3, 1730),
    ('12k_IR021_0.mat',   '209',  12000, 'IR',  'IR',  21,  0, 1797),
    ('12k_IR021_1.mat',   '210',  12000, 'IR',  'IR',  21,  1, 1772),
    ('12k_IR021_2.mat',   '211',  12000, 'IR',  'IR',  21,  2, 1750),
    ('12k_IR021_3.mat',   '212',  12000, 'IR',  'IR',  21,  3, 1730),
    ('12k_IR028_0.mat',   '3001', 12000, 'IR',  'IR',  28,  0, 1797),
    ('12k_IR028_1.mat',   '3002', 12000, 'IR',  'IR',  28,  1, 1772),
    ('12k_IR028_2.mat',   '3003', 12000, 'IR',  'IR',  28,  2, 1750),
    ('12k_IR028_3.mat',   '3004', 12000, 'IR',  'IR',  28,  3, 1730),

    # ── 12k Drive-End — Ball ──────────────────────────────────────────────
    ('12k_B007_0.mat',    '118',  12000, 'Ball','Ball',  7,  0, 1797),
    ('12k_B007_1.mat',    '119',  12000, 'Ball','Ball',  7,  1, 1772),
    ('12k_B007_2.mat',    '120',  12000, 'Ball','Ball',  7,  2, 1750),
    ('12k_B007_3.mat',    '121',  12000, 'Ball','Ball',  7,  3, 1730),
    ('12k_B014_0.mat',    '185',  12000, 'Ball','Ball', 14,  0, 1797),
    ('12k_B014_1.mat',    '186',  12000, 'Ball','Ball', 14,  1, 1772),
    ('12k_B014_2.mat',    '187',  12000, 'Ball','Ball', 14,  2, 1750),
    ('12k_B014_3.mat',    '188',  12000, 'Ball','Ball', 14,  3, 1730),
    ('12k_B021_0.mat',    '222',  12000, 'Ball','Ball', 21,  0, 1797),
    ('12k_B021_1.mat',    '223',  12000, 'Ball','Ball', 21,  1, 1772),
    ('12k_B021_2.mat',    '224',  12000, 'Ball','Ball', 21,  2, 1750),
    ('12k_B021_3.mat',    '225',  12000, 'Ball','Ball', 21,  3, 1730),
    ('12k_B028_0.mat',    '3005', 12000, 'Ball','Ball', 28,  0, 1797),
    ('12k_B028_1.mat',    '3006', 12000, 'Ball','Ball', 28,  1, 1772),
    ('12k_B028_2.mat',    '3007', 12000, 'Ball','Ball', 28,  2, 1750),
    ('12k_B028_3.mat',    '3008', 12000, 'Ball','Ball', 28,  3, 1730),

    # ── 12k Drive-End — Outer Race @6:00 (centered, load zone) ───────────
    ('12k_OR007@6_0.mat', '130',  12000, 'OR',  'OR@6',  7,  0, 1797),
    ('12k_OR007@6_1.mat', '131',  12000, 'OR',  'OR@6',  7,  1, 1772),
    ('12k_OR007@6_2.mat', '132',  12000, 'OR',  'OR@6',  7,  2, 1750),
    ('12k_OR007@6_3.mat', '133',  12000, 'OR',  'OR@6',  7,  3, 1730),
    ('12k_OR014@6_0.mat', '197',  12000, 'OR',  'OR@6', 14,  0, 1797),
    ('12k_OR014@6_1.mat', '198',  12000, 'OR',  'OR@6', 14,  1, 1772),
    ('12k_OR014@6_2.mat', '199',  12000, 'OR',  'OR@6', 14,  2, 1750),
    ('12k_OR014@6_3.mat', '200',  12000, 'OR',  'OR@6', 14,  3, 1730),
    ('12k_OR021@6_0.mat', '234',  12000, 'OR',  'OR@6', 21,  0, 1797),
    ('12k_OR021@6_1.mat', '235',  12000, 'OR',  'OR@6', 21,  1, 1772),
    ('12k_OR021@6_2.mat', '236',  12000, 'OR',  'OR@6', 21,  2, 1750),
    ('12k_OR021@6_3.mat', '237',  12000, 'OR',  'OR@6', 21,  3, 1730),

    # ── 12k Drive-End — Outer Race @3:00 (orthogonal) ────────────────────
    ('12k_OR007@3_0.mat', '144',  12000, 'OR',  'OR@3',  7,  0, 1797),
    ('12k_OR007@3_1.mat', '145',  12000, 'OR',  'OR@3',  7,  1, 1772),
    ('12k_OR007@3_2.mat', '146',  12000, 'OR',  'OR@3',  7,  2, 1750),
    ('12k_OR007@3_3.mat', '147',  12000, 'OR',  'OR@3',  7,  3, 1730),
    ('12k_OR021@3_0.mat', '246',  12000, 'OR',  'OR@3', 21,  0, 1797),
    ('12k_OR021@3_1.mat', '247',  12000, 'OR',  'OR@3', 21,  1, 1772),
    ('12k_OR021@3_2.mat', '248',  12000, 'OR',  'OR@3', 21,  2, 1750),
    ('12k_OR021@3_3.mat', '249',  12000, 'OR',  'OR@3', 21,  3, 1730),

    # ── 12k Drive-End — Outer Race @12:00 (opposite load zone) ───────────
    ('12k_OR007@12_0.mat','156',  12000, 'OR',  'OR@12', 7,  0, 1797),
    ('12k_OR007@12_1.mat','158',  12000, 'OR',  'OR@12', 7,  1, 1772),
    ('12k_OR007@12_2.mat','159',  12000, 'OR',  'OR@12', 7,  2, 1750),
    ('12k_OR007@12_3.mat','160',  12000, 'OR',  'OR@12', 7,  3, 1730),
    ('12k_OR021@12_0.mat','258',  12000, 'OR',  'OR@12',21,  0, 1797),
    ('12k_OR021@12_1.mat','259',  12000, 'OR',  'OR@12',21,  1, 1772),
    ('12k_OR021@12_2.mat','260',  12000, 'OR',  'OR@12',21,  2, 1750),
    ('12k_OR021@12_3.mat','261',  12000, 'OR',  'OR@12',21,  3, 1730),

    # ── 48k Drive-End — Inner Race ────────────────────────────────────────
    ('48k_IR007_0.mat',   '109',  48000, 'IR',  'IR',   7,  0, 1797),
    ('48k_IR007_1.mat',   '110',  48000, 'IR',  'IR',   7,  1, 1772),
    ('48k_IR007_2.mat',   '111',  48000, 'IR',  'IR',   7,  2, 1750),
    ('48k_IR007_3.mat',   '112',  48000, 'IR',  'IR',   7,  3, 1730),
    ('48k_IR014_0.mat',   '174',  48000, 'IR',  'IR',  14,  0, 1797),
    ('48k_IR014_1.mat',   '175',  48000, 'IR',  'IR',  14,  1, 1772),
    ('48k_IR014_2.mat',   '176',  48000, 'IR',  'IR',  14,  2, 1750),
    ('48k_IR014_3.mat',   '177',  48000, 'IR',  'IR',  14,  3, 1730),
    ('48k_IR021_0.mat',   '213',  48000, 'IR',  'IR',  21,  0, 1797),
    ('48k_IR021_1.mat',   '214',  48000, 'IR',  'IR',  21,  1, 1772),
    ('48k_IR021_2.mat',   '215',  48000, 'IR',  'IR',  21,  2, 1750),
    ('48k_IR021_3.mat',   '217',  48000, 'IR',  'IR',  21,  3, 1730),

    # ── 48k Drive-End — Ball ──────────────────────────────────────────────
    ('48k_B007_0.mat',    '122',  48000, 'Ball','Ball',  7,  0, 1797),
    ('48k_B007_1.mat',    '123',  48000, 'Ball','Ball',  7,  1, 1772),
    ('48k_B007_2.mat',    '124',  48000, 'Ball','Ball',  7,  2, 1750),
    ('48k_B007_3.mat',    '125',  48000, 'Ball','Ball',  7,  3, 1730),
    ('48k_B014_0.mat',    '189',  48000, 'Ball','Ball', 14,  0, 1797),
    ('48k_B014_1.mat',    '190',  48000, 'Ball','Ball', 14,  1, 1772),
    ('48k_B014_2.mat',    '191',  48000, 'Ball','Ball', 14,  2, 1750),
    ('48k_B014_3.mat',    '192',  48000, 'Ball','Ball', 14,  3, 1730),
    ('48k_B021_0.mat',    '226',  48000, 'Ball','Ball', 21,  0, 1797),
    ('48k_B021_1.mat',    '227',  48000, 'Ball','Ball', 21,  1, 1772),
    ('48k_B021_2.mat',    '228',  48000, 'Ball','Ball', 21,  2, 1750),
    ('48k_B021_3.mat',    '229',  48000, 'Ball','Ball', 21,  3, 1730),

    # ── 48k Drive-End — Outer Race @6:00 ─────────────────────────────────
    ('48k_OR007@6_0.mat', '135',  48000, 'OR',  'OR@6',  7,  0, 1797),
    ('48k_OR007@6_1.mat', '136',  48000, 'OR',  'OR@6',  7,  1, 1772),
    ('48k_OR007@6_2.mat', '137',  48000, 'OR',  'OR@6',  7,  2, 1750),
    ('48k_OR007@6_3.mat', '138',  48000, 'OR',  'OR@6',  7,  3, 1730),
    ('48k_OR014@6_0.mat', '201',  48000, 'OR',  'OR@6', 14,  0, 1797),
    ('48k_OR014@6_1.mat', '202',  48000, 'OR',  'OR@6', 14,  1, 1772),
    ('48k_OR014@6_2.mat', '203',  48000, 'OR',  'OR@6', 14,  2, 1750),
    ('48k_OR014@6_3.mat', '204',  48000, 'OR',  'OR@6', 14,  3, 1730),
    ('48k_OR021@6_0.mat', '238',  48000, 'OR',  'OR@6', 21,  0, 1797),
    ('48k_OR021@6_1.mat', '239',  48000, 'OR',  'OR@6', 21,  1, 1772),
    ('48k_OR021@6_2.mat', '240',  48000, 'OR',  'OR@6', 21,  2, 1750),
    ('48k_OR021@6_3.mat', '241',  48000, 'OR',  'OR@6', 21,  3, 1730),

    # ── 48k Drive-End — Outer Race @3:00 ─────────────────────────────────
    ('48k_OR007@3_0.mat', '148',  48000, 'OR',  'OR@3',  7,  0, 1797),
    ('48k_OR007@3_1.mat', '149',  48000, 'OR',  'OR@3',  7,  1, 1772),
    ('48k_OR007@3_2.mat', '150',  48000, 'OR',  'OR@3',  7,  2, 1750),
    ('48k_OR007@3_3.mat', '151',  48000, 'OR',  'OR@3',  7,  3, 1730),
    ('48k_OR021@3_0.mat', '250',  48000, 'OR',  'OR@3', 21,  0, 1797),
    ('48k_OR021@3_1.mat', '251',  48000, 'OR',  'OR@3', 21,  1, 1772),
    ('48k_OR021@3_2.mat', '252',  48000, 'OR',  'OR@3', 21,  2, 1750),
    ('48k_OR021@3_3.mat', '253',  48000, 'OR',  'OR@3', 21,  3, 1730),

    # ── 48k Drive-End — Outer Race @12:00 ────────────────────────────────
    ('48k_OR007@12_0.mat','161',  48000, 'OR',  'OR@12', 7,  0, 1797),
    ('48k_OR007@12_1.mat','162',  48000, 'OR',  'OR@12', 7,  1, 1772),
    ('48k_OR007@12_2.mat','163',  48000, 'OR',  'OR@12', 7,  2, 1750),
    ('48k_OR007@12_3.mat','164',  48000, 'OR',  'OR@12', 7,  3, 1730),
    ('48k_OR021@12_0.mat','262',  48000, 'OR',  'OR@12',21,  0, 1797),
    ('48k_OR021@12_1.mat','263',  48000, 'OR',  'OR@12',21,  1, 1772),
    ('48k_OR021@12_2.mat','264',  48000, 'OR',  'OR@12',21,  2, 1750),
    ('48k_OR021@12_3.mat','265',  48000, 'OR',  'OR@12',21,  3, 1730),
]

print(f'Total files to download: {len(FILES)}')
print(f'Breakdown:')
import pandas as pd
df = pd.DataFrame(FILES, columns=['filename','file_id','sample_rate','fault_type','fault_loc','severity','load_hp','rpm'])
print(df.groupby(['sample_rate','fault_loc']).size().to_string())

In [ ]:
# ── Robust download with resume + retry ─────────────────────────────────────

def download_file(fname, file_id, retries=5, wait=3):
    """Download a single file with retry and resume support."""
    url   = BASE + file_id + '.mat'
    fpath = os.path.join(DATA_DIR, fname)
    tmp   = fpath + '.tmp'

    # Already fully downloaded
    if os.path.exists(fpath) and os.path.getsize(fpath) > 5000:
        return 'skip'

    for attempt in range(1, retries + 1):
        try:
            downloaded = os.path.getsize(tmp) if os.path.exists(tmp) else 0
            headers = {'Range': f'bytes={downloaded}-'} if downloaded > 0 else {}

            with requests.get(url, headers=headers, stream=True, timeout=60) as r:
                if r.status_code == 416:          # Already complete
                    os.rename(tmp, fpath)
                    return 'ok'
                r.raise_for_status()
                mode = 'ab' if downloaded > 0 else 'wb'
                with open(tmp, mode) as f:
                    for chunk in r.iter_content(chunk_size=65536):
                        if chunk:
                            f.write(chunk)

            os.rename(tmp, fpath)
            return 'ok'

        except Exception as e:
            if attempt < retries:
                print(f'  Retry {attempt}/{retries} for {fname} — {type(e).__name__}')
                time.sleep(wait * attempt)
            else:
                return 'fail'

# ── Run downloads ────────────────────────────────────────────────────────────
results = {'ok': [], 'skip': [], 'fail': []}

for fname, file_id, *_ in tqdm(FILES, desc='Downloading'):
    r = download_file(fname, file_id)
    results[r].append(fname)

print(f"\n✅ Downloaded : {len(results['ok'])}")
print(f"⏭  Skipped    : {len(results['skip'])} (already present)")
print(f"❌ Failed     : {len(results['fail'])}")
if results['fail']:
    print('\nFailed files (re-run cell to retry):')
    for f in results['fail']:
        print(f'  {f}')

In [ ]:
# ── Verify all files and save manifest CSV ───────────────────────────────────
import pandas as pd
from scipy.io import loadmat

def probe_signal_length(fpath):
    """Load mat file and return the DE_time signal length."""
    try:
        mat = loadmat(fpath)
        key = next((k for k in mat if 'DE_time' in k), None)
        if key:
            return len(mat[key].flatten())
        # Fallback: first non-meta key
        keys = [k for k in mat if not k.startswith('_')]
        return len(mat[keys[0]].flatten()) if keys else 0
    except:
        return 0

manifest_rows = []
for fname, file_id, sample_rate, fault_type, fault_loc, severity, load_hp, rpm in FILES:
    fpath   = os.path.join(DATA_DIR, fname)
    present = os.path.exists(fpath) and os.path.getsize(fpath) > 5000
    sig_len = probe_signal_length(fpath) if present else 0
    manifest_rows.append({
        'filename':    fname,
        'file_id':     file_id,
        'sample_rate': sample_rate,
        'fault_type':  fault_type,
        'fault_loc':   fault_loc,
        'severity':    severity,
        'load_hp':     load_hp,
        'rpm':         rpm,
        'signal_len':  sig_len,
        'present':     present,
    })

df_manifest = pd.DataFrame(manifest_rows)
df_manifest.to_csv('cwru_manifest.csv', index=False)

print('=== Download Manifest ===')
print(f'Total files registered : {len(df_manifest)}')
print(f'Files present on disk  : {df_manifest["present"].sum()}')
print(f'Missing                : {(~df_manifest["present"]).sum()}')
print()
print('Files by sample rate and fault location:')
print(df_manifest[df_manifest['present']].groupby(['sample_rate','fault_loc']).size().to_string())
print()
print('Signal length summary (samples per file):')
print(df_manifest[df_manifest['present']].groupby('sample_rate')['signal_len'].describe().to_string())
print()
print('Manifest saved → cwru_manifest.csv')

In [ ]:
# ── Quick sanity check: inspect one file from each rate ─────────────────────
import numpy as np

for sample_file in ['12k_IR007_0.mat', '48k_IR007_0.mat', 'Normal_0.mat']:
    fpath = os.path.join(DATA_DIR, sample_file)
    if not os.path.exists(fpath):
        print(f'  {sample_file} — NOT FOUND')
        continue
    mat  = loadmat(fpath)
    keys = [k for k in mat if not k.startswith('_')]
    de_key = next((k for k in keys if 'DE_time' in k), keys[0])
    sig  = mat[de_key].flatten()
    print(f'{sample_file}')
    print(f'  Keys       : {keys}')
    print(f'  DE key     : {de_key}')
    print(f'  Signal len : {len(sig):,} samples')
    print(f'  Min/Max    : {sig.min():.4f} / {sig.max():.4f}')
    print(f'  File size  : {os.path.getsize(fpath)/1024:.1f} KB')
    print()